**Trade-off Analysis: 7 Abordagens de Agregação de Chunks**

Este notebook compara sete abordagens de processamento de texto para fine-tuning do modelo LegalBERT:
- **Truncation (512)**: Trunca o texto em 512 tokens (baseline)
- **Mean Pooling - Stride 0**: Chunks sem sobreposição + média
- **Mean Pooling - Stride 128**: Chunks com 384 tokens de sobreposição + média
- **Mean Pooling - Stride 256**: Chunks com 256 tokens de sobreposição + média
- **Max Pooling - Stride 0**: Chunks sem sobreposição + máximo
- **Max Pooling - Stride 128**: Chunks com 384 tokens de sobreposição + máximo
- **Max Pooling - Stride 256**: Chunks com 256 tokens de sobreposição + máximo

## ⚡ Documentação de Otimizações de Performance

Este notebook foi otimizado para maximizar a utilização de hardware de alta performance (como GPUs **NVIDIA A100**) e evitar gargalos (bottlenecks) de CPU durante o treinamento e a inferáncia.

### 1. Otimizações de GPU (A100 & Tensor Cores)
- **TF32 (TensorFloat-32):** Habilitado via `torch.backends.cuda.matmul.allow_tf32` e `torch.backends.cudnn.allow_tf32`. A arquitetura Ampere (A100) possui Tensor Cores dedicados que aceleram operações matriciais massivamente utilizando TF32 sem perda perceptível de precisão.
- **AMP (Automatic Mixed Precision):** Utilizado nativamente nos loops de treino/inferáncia (`torch.autocast`), reduzindo o consumo de memória pela metade (Float16) e acelerando o cálculo dos gradientes.

### 2. Dataloader e Mitigação de Data Starvation
- **Móltiplos Workers (`NUM_WORKERS`):** Configurado dinamicamente para usar o máximo de núcleos disponíveis na CPU (`os.cpu_count()`). Isso garante que os batches sejam tokenizados e preparados rapidamente.
- **Prefetch Factor:** Aumentado de `2` para `4` no `DataLoader`. Isso obriga a CPU a deixar mais batches prontos na fila (RAM) enquanto a GPU processa o batch atual, evitando que a GPU fique ociosa.
- **Persistent Workers:** Mantido ativo (`persistent_workers=True`) para evitar o overhead de recriação dos processos a cada época.

### 3. Ajuste de Hiperparâmetros, Literatura e Congelamento de Camadas
- **Batch Size Limitado a 32:** Apesar da A100 comportar batches muito maiores, a literatura científica de treinamento com base no BERT estabelece que os melhores índices de generalização ocorrem com **Batch Size 16 ou 32**.
- **Feature Extraction (Transfer Learning):** Como o parâmetro `NUM_LAYERS = 12` congela todas as 12 camadas do modelo original, a tarefa executada não é de *Full Fine-Tuning*, mas sim de treinar uma nova cabeça de classificação profunda do zero. Devido a isso, é esperado um maior número de épocas de convergáncia. A paciáncia do modelo (`EARLY_STOPPING`) está mantida em **5**, mas poderia ser estendida para até 10 para compensar o início de aprendizado mais lento da cabeça extratora.

### Import Libraries and Set the intial variables

In [ ]:
#!pip install transformers

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from pylab import rcParams

from sklearn.metrics import confusion_matrix, classification_report

# Torch ML libraries
from transformers import BertModel, AutoTokenizer, get_linear_schedule_with_warmup
import torch

from torch import nn
from torch.utils.data import Dataset, DataLoader, Sampler
# Misc.
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Set intial variables and constants
%config InlineBackend.figure_format='retina'

# Graph Designs
sns.set(style='whitegrid', palette='muted', font_scale=1.2)
HAPPY_COLORS_PALETTE = ["#01BEFE", "#FFDD00", "#FF7D00", "#FF006D", "#ADFF02", "#8F00FF"]
sns.set_palette(sns.color_palette(HAPPY_COLORS_PALETTE))
rcParams['figure.figsize'] = 12, 8

# Random seed for reproducibilty
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

# Set GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# GPU Optimizations
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print(f"Acelerador de Hardware Detectado: {gpu_name}")
    if "A100" in gpu_name:
        print("Habilitando TF32 para otimização de tensores (exclusivo Ampere+)...")
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
    elif "T4" in gpu_name:
        print("Arquitetura Turing detectada. Utilizaremos AMP (FP16) via autocast para máxima performance nos Tensor Cores.")


##Parametrização conforme ambiente de execução.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import torch

BASE_PATH_GENERAL = '/content/drive/MyDrive/_Mestrado PUCPR/'
BASE_PATH_DATASET = f'{BASE_PATH_GENERAL}Datasets/'
BASE_MODEL_SAVED = f'{BASE_PATH_GENERAL}Datasets/models/'

MODEL_CHECKPOINT = 'raquelsilveira/legalbertpt_fp'
NUM_LAYERS = 12

RANDOM_STATE = 42
EPOCHS = 100
APPROACH_EXECUTION = 'epoch' #'step'
LEARNING_RATE = 1e-4 # Recomendado para fine-tuning (5e-5, 3e-5 ou 2e-5), mas para treinamento apenas da camada densa, pode ser ajustado para 1e-4.

BATCH_SIZE = 8 # (Devlin et al., 2018)

WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
EARLY_STOPPING = 5 # Como todas as camadas do BERT estao congeladas, o treinamento requer mais epocas. 5 a 10 e recomendado.
USE_BUCKETED_BATCHING = True  # Agrupa documentos por num_chunks para reduzir padding no treino
BUCKET_SIZE_MULTIPLIER = 50   # Tamanho do bucket = batch_size * multiplicador
class_names = ['Desfavorável ao Réu', 'Favorável ao Réu']

CHUNK_SIZE = 512
EFFECTIVE_CHUNK_TOKENS = CHUNK_SIZE - 2

MAX_CHUNKS = None
MIN_CHUNK_TOKENS = 8
NROWS_LOADING = None

def get_safe_label(label):
    return label.replace(" ", "_").replace("(", "").replace(")", "").replace(",", "").replace("=", "")

# Reduzir NUM_WORKERS da G4 (T4) para evitar gargalos na CPU menor
NUM_WORKERS = min(os.cpu_count() or 2, 4)

In [ ]:
FORCE_RETRAIN = False  # True: força o treinamento (caso exista, sobreescreve); False: apenas carrega modelos salvos e roda inferencias (senão existe, gera ele)
#STRATEGY = 'LONG' # "LONG": Textos Longos (Acima de 512) | "SMALL": Textos até 512 | "MIX": Combinação de LONG e SMALL | LONG-ACORDAO | LONG-DECISAO | SMALL-ACORDAO
STRATEGY = 'SMALL-ACORDAO'
base_model_name = 'legalbertpt_fp'

APPROACHES = [
    {
        'label': 'Truncation (512)',
        'stride': CHUNK_SIZE,
        'max_chunks': 1,
        'batch_size': BATCH_SIZE,
        'pooling': 'max',
        'min_chunk_tokens': 1,
        'enabled': True,
        'model_name': f'best_model_{base_model_name}_{get_safe_label("Truncation (512)")}_{STRATEGY}.bin'
    },
    {
        'label': 'Mean Pooling - Stride 0',
        'stride': CHUNK_SIZE,
        'max_chunks': MAX_CHUNKS,
        'batch_size': BATCH_SIZE,
        'pooling': 'mean',
        'min_chunk_tokens': MIN_CHUNK_TOKENS,
        'enabled': False,
        'model_name': f'best_model_{base_model_name}_{get_safe_label("Mean Pooling - Stride 0")}_{STRATEGY}.bin'
    },
    {
        'label': 'Mean Pooling - Stride 64',
        'stride': 64,
        'max_chunks': MAX_CHUNKS,
        'batch_size': BATCH_SIZE,
        'pooling': 'mean',
        'min_chunk_tokens': MIN_CHUNK_TOKENS,
        'enabled': False,
        'model_name': f'best_model_{base_model_name}_{get_safe_label("Mean Pooling - Stride 64")}_{STRATEGY}.bin'
    },
    {
        'label': 'Mean Pooling - Stride 128',
        'stride': 128,
        'max_chunks': MAX_CHUNKS,
        'batch_size': BATCH_SIZE,
        'pooling': 'mean',
        'min_chunk_tokens': MIN_CHUNK_TOKENS,
        'enabled': False,
        'model_name': f'best_model_{base_model_name}_{get_safe_label("Mean Pooling - Stride 128")}_{STRATEGY}.bin'
    },
    {
        'label': 'Mean Pooling - Stride 256',
        'stride': 256,
        'max_chunks': MAX_CHUNKS,
        'batch_size': BATCH_SIZE,
        'pooling': 'mean',
        'min_chunk_tokens': MIN_CHUNK_TOKENS,
        'enabled': False,
        'model_name': f'best_model_{base_model_name}_{get_safe_label("Mean Pooling - Stride 256")}_{STRATEGY}.bin'
    },
    {
        'label': 'Max Pooling - Stride 0',
        'stride': CHUNK_SIZE,
        'max_chunks': MAX_CHUNKS,
        'batch_size': BATCH_SIZE,
        'pooling': 'max',
        'min_chunk_tokens': MIN_CHUNK_TOKENS,
        'enabled': False,
        'model_name': f'best_model_{base_model_name}_{get_safe_label("Max Pooling - Stride 0")}_{STRATEGY}.bin'
    },
    {
        'label': 'Max Pooling - Stride 64',
        'stride': 64,
        'max_chunks': MAX_CHUNKS,
        'batch_size': BATCH_SIZE,
        'pooling': 'max',
        'min_chunk_tokens': MIN_CHUNK_TOKENS,
        'enabled': False,
        'model_name': f'best_model_{base_model_name}_{get_safe_label("Max Pooling - Stride 64")}_{STRATEGY}.bin'
    },
    {
        'label': 'Max Pooling - Stride 128',
        'stride': 128,
        'max_chunks': MAX_CHUNKS,
        'batch_size': BATCH_SIZE,
        'pooling': 'max',
        'min_chunk_tokens': MIN_CHUNK_TOKENS,
        'enabled': False,
        'model_name': f'best_model_{base_model_name}_{get_safe_label("Max Pooling - Stride 128")}_{STRATEGY}.bin'
    },
    {
        'label': 'Max Pooling - Stride 256',
        'stride': 256,
        'max_chunks': MAX_CHUNKS,
        'batch_size': BATCH_SIZE,
        'pooling': 'max',
        'min_chunk_tokens': MIN_CHUNK_TOKENS,
        'enabled': False,
        'model_name': f'best_model_{base_model_name}_{get_safe_label("Max Pooling - Stride 256")}_{STRATEGY}.bin'
    },
    {
        'label': 'Head Tail - 128-382',
        'stride': 512,
        'max_chunks': 1,
        'batch_size': BATCH_SIZE,
        'pooling': None,
        'min_chunk_tokens': (128, 382),  # (head_size, tail_size) -> ativa modo Head-Tail
        'enabled': True,
        'model_name': f'best_model_{base_model_name}_{get_safe_label("Head Tail - 128-382")}_{STRATEGY}.bin'
    },
    {
        'label': 'Head Tail - 256-254',
        'stride': 512,
        'max_chunks': 1,
        'batch_size': BATCH_SIZE,
        'pooling': None,
        'min_chunk_tokens': (256, 254),  # (head_size, tail_size) -> ativa modo Head-Tail
        'enabled': True,
        'model_name': f'best_model_{base_model_name}_{get_safe_label("Head Tail - 256-254")}_{STRATEGY}.bin'
    }
]

In [ ]:
import os

# Paths padrão para reprodutibilidade
if STRATEGY == 'MIX':
  train_path = f"{BASE_PATH_DATASET}preprocessed_acordaos_HC_under_2021_2025_train-punct.csv"
  test_path = f"{BASE_PATH_DATASET}preprocessed_acordaos_HC_under_2021_2025_test-punct.csv"
elif STRATEGY == 'SMALL':
  train_path = f"{BASE_PATH_DATASET}preprocessed_acordaos_HC_under_2021_2025_train_small-punct.csv"
  test_path = f"{BASE_PATH_DATASET}preprocessed_acordaos_HC_under_2021_2025_test_small-punct.csv"
elif STRATEGY == 'LONG':
  train_path = f"{BASE_PATH_DATASET}preprocessed_HC_under_2021_2025_train_long_full-text.csv"
  test_path = f"{BASE_PATH_DATASET}preprocessed_HC_under_2021_2025_test_long_full-text.csv"
elif STRATEGY == 'LONG-ACORDAO':
  train_path = f"{BASE_PATH_DATASET}preprocessed_acordaos_HC_under_2021_2025_train_long-punct.csv"
  test_path = f"{BASE_PATH_DATASET}preprocessed_acordaos_HC_under_2021_2025_test_long-punct.csv"
elif STRATEGY == 'LONG-DECISAO':
  train_path = f"{BASE_PATH_DATASET}preprocessed_decisoes_HC_under_2021_2025_train_long-punct.csv"
  test_path = f"{BASE_PATH_DATASET}preprocessed_decisoes_HC_under_2021_2025_test_long-punct.csv"
elif STRATEGY == 'SMALL-ACORDAO':
  train_path = f"{BASE_PATH_DATASET}preprocessed_acordaos_HC_under_2021_2025_train_small-punct.csv"
  test_path = f"{BASE_PATH_DATASET}preprocessed_acordaos_HC_under_2021_2025_test_small-punct.csv"
else:
  raise Exception("Strategy type is not poised for this train.")

# Apenas carrega os arquivos já persistidos (sem split)
if not os.path.exists(train_path):
    raise FileNotFoundError(f"Arquivo de treino não encontrado: {train_path}")
if not os.path.exists(test_path):
    raise FileNotFoundError(f"Arquivo de teste não encontrado: {test_path}")

print("Carregando arquivos de treino e teste persistidos...")
df_train = pd.read_csv(train_path, sep=';', nrows=NROWS_LOADING)
df_test = pd.read_csv(test_path, sep=';', nrows=NROWS_LOADING*0.2 if NROWS_LOADING else NROWS_LOADING)

print(f"Treino carregado de: {train_path} | shape: {df_train.shape}")
print(f"Teste carregado de: {test_path} | shape: {df_test.shape}")

In [ ]:
# Function para converter a decisão em valor inteiro (0 = Desfavorável, 1 = Favorável)
def cast_decisao_to_int(decisao_favoravel):
    if decisao_favoravel == 'S':
        return 1
    else:
        return 0

# Apply to train and test datasets
if 'DECISAO_FAVORAVEL' in df_train.columns and 'classe' not in df_train.columns:
    df_train['classe'] = df_train.DECISAO_FAVORAVEL.apply(cast_decisao_to_int)
if 'DECISAO_FAVORAVEL' in df_test.columns and 'classe' not in df_test.columns:
    df_test['classe'] = df_test.DECISAO_FAVORAVEL.apply(cast_decisao_to_int)

print(f"Classes no treino: {df_train['classe'].value_counts().to_dict()}")
print(f"Classes no teste:  {df_test['classe'].value_counts().to_dict()}")

# BERT

### Preparing Torch Dataset

###  `HCAcordaoSTJ` (Custom Dataset com Sliding Window)

In [ ]:
from functools import partial
from typing import Dict, List, Union, Iterator, Tuple
import math
import numpy as np
import torch
from torch.utils.data import Dataset
from torch.nn.utils.rnn import pad_sequence
class HCAcordaoSTJ(Dataset):
    """Dataset com Sliding Window para textos longos e suporte a Head-Tail."""
    def __init__(self, reports: np.ndarray, targets: np.ndarray, tokenizer,
                 chunk_size: int, stride: int, max_chunks: int = None,
                 min_chunk_tokens: Union[int, Tuple[int, int]] = 1) -> None:
        self.reports = reports.astype(str)
        self.targets = targets
        self.chunk_size = chunk_size
        self.stride = stride
        self.max_chunks = max_chunks
        self.pad_token_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0
        self.doc_chunk_counts: List[int] = []
        # Tokens especiais [CLS] (id=101) e [SEP] (id=102) ocupam 2 posições.
        # effective_length = tokens de CONTEÚDO por chunk = chunk_size - 2
        # Ex.: chunk_size=512  ?  effective_length=510 tokens de conteúdo
        #      input final:  [CLS] + 510 tokens + [SEP]  =  512  (limite do modelo)
        self.effective_length = chunk_size - 2

        # Detecta modo Head-Tail: se min_chunk_tokens for tupla (head_size, tail_size)
        self.head_tail_mode = isinstance(min_chunk_tokens, tuple)
        if self.head_tail_mode:
            self.head_size, self.tail_size = min_chunk_tokens
            self.min_chunk_tokens = 1  # não usado em head-tail
            print(f"  [Dataset] Modo HEAD-TAIL ativado: head={self.head_size}, tail={self.tail_size}")
        else:
            self.head_size, self.tail_size = None, None
            self.min_chunk_tokens = max(1, min_chunk_tokens)

        # Tokeniza sem truncar para obter todos os tokens
        all_encodings = tokenizer(
            self.reports.tolist(),
            add_special_tokens=False,
            return_attention_mask=False,
        )
        # Cria chunks por documento
        self.all_input_ids: List[List[torch.Tensor]] = []
        self.all_attention_masks: List[List[torch.Tensor]] = []
        cls_id = tokenizer.cls_token_id
        sep_id = tokenizer.sep_token_id
        # Contadores para relatório de impacto do filtro min_chunk_tokens
        total_chunks_before_filter = 0
        total_chunks_after_filter = 0
        total_chunks_after_max = 0
        for doc_tokens in all_encodings['input_ids']:
            chunks_ids = []
            chunks_mask = []
            doc_chunks_before = 0

            # ========== MODO HEAD-TAIL ==========
            if self.head_tail_mode:
                doc_chunks_before = 1
                # Se texto é curto (cabe em effective_length), usa todo
                if len(doc_tokens) <= self.effective_length:
                    content_tokens = doc_tokens
                else:
                    # Pega head (início) + tail (fim)
                    head_tokens = doc_tokens[:self.head_size]
                    tail_tokens = doc_tokens[-self.tail_size:]
                    content_tokens = head_tokens + tail_tokens

                # Monta input: [CLS] + content + [SEP]
                input_ids = [cls_id] + content_tokens + [sep_id]
                attention_mask = [1] * len(input_ids)

                # Padding até chunk_size se necessário
                pad_len = self.chunk_size - len(input_ids)
                if pad_len > 0:
                    input_ids = input_ids + [self.pad_token_id] * pad_len
                    attention_mask = attention_mask + [0] * pad_len

                chunks_ids.append(torch.tensor(input_ids, dtype=torch.long))
                chunks_mask.append(torch.tensor(attention_mask, dtype=torch.long))

            # ========== MODO SLIDING WINDOW (original) ==========
            else:
                # Gera janelas deslizantes
                start = 0
                while start < len(doc_tokens):
                    end = start + self.effective_length
                    window_tokens = doc_tokens[start:end]
                    doc_chunks_before += 1
                    # Descarta chunks muito curtos quando ja existem chunks validos.
                    # Fallback: se o documento for muito curto, mantemos ao menos 1 chunk.
                    if len(window_tokens) < self.min_chunk_tokens and len(chunks_ids) > 0:
                        if end >= len(doc_tokens):
                            break
                        start += self.stride
                        continue
                    # Adiciona [CLS] ... tokens ... [SEP]
                    input_ids = [cls_id] + window_tokens + [sep_id]
                    attention_mask = [1] * len(input_ids)
                    chunks_ids.append(torch.tensor(input_ids, dtype=torch.long))
                    chunks_mask.append(torch.tensor(attention_mask, dtype=torch.long))
                    # Se já cobriu todo o documento, para
                    if end >= len(doc_tokens):
                        break
                    start += self.stride

            total_chunks_before_filter += doc_chunks_before
            total_chunks_after_filter += len(chunks_ids)
            # Limita número de chunks se configurado
            if self.max_chunks is not None:
                chunks_ids = chunks_ids[:self.max_chunks]
                chunks_mask = chunks_mask[:self.max_chunks]
            total_chunks_after_max += len(chunks_ids)
            # Garante ao menos um chunk por documento
            if len(chunks_ids) == 0:
                window_tokens = doc_tokens[:self.effective_length]
                input_ids = [cls_id] + window_tokens + [sep_id]
                attention_mask = [1] * len(input_ids)
                chunks_ids.append(torch.tensor(input_ids, dtype=torch.long))
                chunks_mask.append(torch.tensor(attention_mask, dtype=torch.long))
            self.all_input_ids.append(chunks_ids)
            self.all_attention_masks.append(chunks_mask)
            self.doc_chunk_counts.append(len(chunks_ids))
        # --- Relatório de impacto do filtro min_chunk_tokens ---
        n_docs = len(self.reports)
        avg_before = total_chunks_before_filter / n_docs if n_docs > 0 else 0.0
        avg_after_filter = total_chunks_after_filter / n_docs if n_docs > 0 else 0.0
        avg_after_max = total_chunks_after_max / n_docs if n_docs > 0 else 0.0
        removed = total_chunks_before_filter - total_chunks_after_filter
        pct_removed = 100.0 * removed / total_chunks_before_filter if total_chunks_before_filter > 0 else 0.0
        print(f"  [Dataset] Documentos: {n_docs}")
        if self.head_tail_mode:
            print(f"  [Dataset] Modo Head-Tail: 1 chunk/doc (head={self.head_size} + tail={self.tail_size} = {self.head_size + self.tail_size} tokens)")
        else:
            print(f"  [Dataset] Chunks/doc antes do filtro (min={self.min_chunk_tokens}):  {avg_before:.2f}  (total={total_chunks_before_filter})")
            print(f"  [Dataset] Chunks/doc após  filtro (min={self.min_chunk_tokens}):  {avg_after_filter:.2f}  (removidos={removed}, {pct_removed:.1f}%)")
        if self.max_chunks is not None:
            print(f"  [Dataset] Chunks/doc após max_chunks={self.max_chunks}:  {avg_after_max:.2f}  (total={total_chunks_after_max})")
        else:
            print(f"  [Dataset] max_chunks=None → sem limitação adicional")
    def __len__(self) -> int:
        return len(self.reports)
    def __getitem__(self, item: int) -> Dict[str, Union[str, torch.Tensor, List[torch.Tensor]]]:
        return {
            'report_text': self.reports[item],
            'input_ids': self.all_input_ids[item],
            'attention_mask': self.all_attention_masks[item],
            'num_chunks': len(self.all_input_ids[item]),
            'targets': torch.tensor(self.targets[item], dtype=torch.long)
        }
def sliding_window_collate_fn(batch: List[Dict], pad_token_id: int = 0) -> Dict[str, Union[torch.Tensor, List[str]]]:
    """Collate function com padding dinâmico para batches com número variável de chunks por documento."""
    all_ids, all_masks, targets, texts = [], [], [], []
    chunk_counts = []
    for sample in batch:
        all_ids.extend(sample['input_ids'])
        all_masks.extend(sample['attention_mask'])
        targets.append(sample['targets'])
        texts.append(sample['report_text'])
        chunk_counts.append(sample['num_chunks'])
    # Padding dinâmico at? o maior chunk do batch atual
    input_ids = pad_sequence(all_ids, batch_first=True, padding_value=pad_token_id)
    attention_mask = pad_sequence(all_masks, batch_first=True, padding_value=0)
    targets = torch.stack(targets)
    chunk_counts = torch.tensor(chunk_counts, dtype=torch.long)
    return {
        'input_ids': input_ids,
        'attention_mask': attention_mask,
        'targets': targets,
        'chunk_counts': chunk_counts,
        'report_text': texts
    }


class BucketByChunkCountBatchSampler(Sampler[List[int]]):
    """Batch sampler que agrupa amostras com num_chunks semelhante."""

    def __init__(self, lengths: List[int], batch_size: int, shuffle: bool,
                 bucket_size_multiplier: int = 50, drop_last: bool = False,
                 seed: int = 42) -> None:
        self.lengths = lengths
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.drop_last = drop_last
        self.seed = seed
        self.epoch = 0
        self.bucket_size = max(batch_size, batch_size * max(1, bucket_size_multiplier))

    def __iter__(self) -> Iterator[List[int]]:
        n = len(self.lengths)
        if n == 0:
            return

        generator = torch.Generator()
        generator.manual_seed(self.seed + self.epoch)

        if self.shuffle:
            indices = torch.randperm(n, generator=generator).tolist()
        else:
            indices = list(range(n))

        batches: List[List[int]] = []
        for i in range(0, n, self.bucket_size):
            bucket = indices[i:i + self.bucket_size]
            bucket.sort(key=lambda idx: self.lengths[idx], reverse=True)

            for j in range(0, len(bucket), self.batch_size):
                batch = bucket[j:j + self.batch_size]
                if len(batch) == self.batch_size or not self.drop_last:
                    batches.append(batch)

        if self.shuffle and len(batches) > 1:
            order = torch.randperm(len(batches), generator=generator).tolist()
            batches = [batches[k] for k in order]

        self.epoch += 1
        for batch in batches:
            yield batch

    def __len__(self) -> int:
        n = len(self.lengths)
        if self.drop_last:
            return n // self.batch_size
        return math.ceil(n / self.batch_size)

In [ ]:
if 'df_train' not in globals() or 'df_test' not in globals():
    raise RuntimeError("Execute primeiro a célula de carga inteligente para criar/carregar treino e teste.")

print(f"Train shape: {df_train.shape}")
print(f"Test shape: {df_test.shape}")

In [ ]:
import os

def create_data_loader(df: pd.DataFrame, tokenizer, chunk_size: int, stride: int,
                       batch_size: int, max_chunks: int = None,
                       min_chunk_tokens: Union[int, Tuple[int, int]] = 1, shuffle: bool = False,
                       use_bucketed_batching: bool = USE_BUCKETED_BATCHING,
                       bucket_size_multiplier: int = BUCKET_SIZE_MULTIPLIER) -> DataLoader:
    ds = HCAcordaoSTJ(
        reports=df.RELATORIO.to_numpy(),
        targets=df.classe.to_numpy(),
        tokenizer=tokenizer,
        chunk_size=chunk_size,
        stride=stride,
        max_chunks=max_chunks,
        min_chunk_tokens=min_chunk_tokens
    )

    workers = min(NUM_WORKERS, os.cpu_count() or 1)

    loader_kwargs = {
        'dataset': ds,
        'num_workers': workers,
        'pin_memory': (device.type == 'cuda'),
        'collate_fn': partial(sliding_window_collate_fn, pad_token_id=ds.pad_token_id)
    }

    # Bucketed batching é aplicado no treino (shuffle=True)
    if use_bucketed_batching and shuffle:
        loader_kwargs['batch_sampler'] = BucketByChunkCountBatchSampler(
            lengths=ds.doc_chunk_counts,
            batch_size=batch_size,
            shuffle=True,
            bucket_size_multiplier=bucket_size_multiplier,
            drop_last=False,
            seed=RANDOM_SEED,
        )
    else:
        loader_kwargs['batch_size'] = batch_size
        loader_kwargs['shuffle'] = shuffle

    if workers > 0:
        loader_kwargs['persistent_workers'] = True
        # Aumentado de 2 para 4 para empilhar mais dados para a GPU, evitando starvation.
        loader_kwargs['prefetch_factor'] = 4

    return DataLoader(**loader_kwargs)


## ⚖️ Arquitetura do Modelo e Fundamentação Científica

As escolhas de hiperparâmetros e o design da arquitetura foram estritamente pautados nas diretrizes da literatura científica pertinente ao treinamento de modelos da família BERT (*Devlin et al., 2018*) e nas particularidades do domínio jurídico.

### 1. Congelamento do BERT (Feature Extraction / Linear Probing)
O parâmetro `NUM_LAYERS = 12` configura o modelo para **congelar todas as 12 camadas do encoder do BERT**. Em vez de aplicar um *Full Fine-Tuning* (atualizar todos os pesos), a estratégia adotada é de **Feature Extraction** (ou *Linear Probing* estendido).
* **Prevenção de Overfitting e Catastrophic Forgetting:** Como modelos baseados em BERT (como o LegalBERT) foram pré-treinados em bilhões de tokens, o *Full Fine-Tuning* em textos longos e ruidosos (ex: relatórios de acórdãos) gera alto risco de o modelo memorizar peculiaridades da base de treino (*overfitting*) ou esquecer a semântica jurídica geral adquirida previamente (*catastrophic forgetting*).
* **Estabilidade e Respaldo Científico:** Conforme a literatura recente, enquanto o *Full Fine-Tuning* pode ser instável e sensível, a abordagem de congelar a base e treinar uma MLP robusta oferece um treinamento mais seguro e generalizável, alinhando-se aos princípios das tendências atuais de *Parameter-Efficient Fine-Tuning (PEFT)*.

### 2. Cabeça de Classificação (Fully Connected Layers)
Como os embeddings do BERT estão congelados, a responsabilidade de aprender a tarefa downstream recai inteiramente sobre uma nova cabeça de classificação acoplada ao topo do modelo. Foi desenhada uma MLP (Multi-Layer Perceptron) profunda e expressiva:
* **Topologia:** 4 camadas ocultas densas expansivas (`5000 -> 2500 -> 1250 -> 750 -> n_classes`).
* **Regularização:** Uso de funcões de ativação **ReLU** e **Dropout (p=0.3)** em cada camada para garantir robustez aos ruídos dos textos longos.

### 3. Decisões Baseadas na Literatura e Hiperparâmetros
* **Batch Size (32):** A literatura recomenda expressamente lotes de tamanho **16 ou 32** para tarefas baseadas no BERT. Mesmo com hardware superdimensionado (NVIDIA A100), o uso de batches massivos é evitado para preservar as propriedades intrínsecas de regularização estocástica (evitando convergir para *sharp minimas*).
* **Learning Rate (1e-4):** Como as 12 camadas do BERT foram congeladas (*Feature Extraction*) e estamos treinando apenas a nova cabeça de classificação do zero, uma taxa de aprendizado maior (`1e-4`) é a ideal. Em cenários de *Full Fine-Tuning*, utilizaríamos taxas menores (como `2e-5` ou `3e-5`) para não destruir os pesos previamente aprendidos do modelo-base.
* **Early Stopping e Épocas:** Enquanto o *Fine-Tuning* padrão converge em 2 a 4 épocas, o treinamento de uma estrutura Fully Connected «do zero» sobre um BERT congelado demanda um aprendizado gradual. Assim, o `EARLY_STOPPING` foi ampliado (5) para dar à MLP tempo hábil de decifrar as extrações multidimensionais feitas pelo Transformer antes de considerar a interrupção do treinamento.

In [ ]:
class HCAcordaoClassifier(nn.Module):

    # Mapa de arquitetura da cabeça classificadora por número de camadas do modelo
    HEAD_CONFIG = {
        12: (5_000, 2_500, 1_250, 750),
    }

    def __init__(self, n_classes: int, num_layers: int, model_name: str, pooling_strategy: str = 'max') -> None:
        super(HCAcordaoClassifier, self).__init__()
        self.bert = BertModel.from_pretrained(model_name)
        self.pooling_strategy = pooling_strategy

        if self.pooling_strategy not in {'max', 'mean'}:
            raise ValueError(f"pooling_strategy invalida ({self.pooling_strategy}). Use 'max' ou 'mean'.")

        if num_layers not in self.HEAD_CONFIG:
            raise ValueError(f"num_layers inválido ({num_layers}). Valores suportados: {list(self.HEAD_CONFIG.keys())}")

        h1, h2, h3, h4 = self.HEAD_CONFIG[num_layers]

        self.fc1 = nn.Linear(self.bert.config.hidden_size, h1)
        self.drop1 = nn.Dropout(p=0.3)
        self.relu1 = nn.ReLU()

        self.fc2 = nn.Linear(h1, h2)
        self.drop2 = nn.Dropout(p=0.3)
        self.relu2 = nn.ReLU()

        self.fc3 = nn.Linear(h2, h3)
        self.drop3 = nn.Dropout(p=0.3)
        self.relu3 = nn.ReLU()

        self.fc4 = nn.Linear(h3, h4)
        self.drop4 = nn.Dropout(p=0.3)
        self.relu4 = nn.ReLU()

        self.out = nn.Linear(h4, n_classes)

        self.freeze_layers(num_layers)

    def freeze_layers(self, num_layers_to_freeze: int) -> None:
        print(f"Freezing embeddings + first {num_layers_to_freeze} encoder layers...")
        for name, param in self.bert.named_parameters():
            if 'embeddings' in name:
                param.requires_grad = False
            elif 'encoder.layer' in name:
                layer_num = int(name.split('.')[2])
                if layer_num < num_layers_to_freeze:
                    param.requires_grad = False
                else:
                    param.requires_grad = True
            else:
                param.requires_grad = True
        print("Layers freezing complete.")

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor,
                chunk_counts: torch.Tensor = None) -> torch.Tensor:
        """Forward pass com suporte a Sliding Window."""
        output = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        chunk_embeddings = output.pooler_output

        # Agrega chunks por documento se sliding window está ativo
        if chunk_counts is not None and chunk_counts.sum() != chunk_counts.shape[0]:
            # Agrupa os embeddings por documento e aplica a estratégia de pooling
            doc_embeddings = []
            idx = 0
            for count in chunk_counts:
                c = count.item()
                doc_chunks = chunk_embeddings[idx:idx + c]
                if self.pooling_strategy == 'max':
                    pooled, _ = doc_chunks.max(dim=0)
                else:
                    pooled = doc_chunks.mean(dim=0)
                doc_embeddings.append(pooled)
                idx += c
            hidden = torch.stack(doc_embeddings)
        else:
            hidden = chunk_embeddings

        hidden = self.fc1(hidden)
        hidden = self.relu1(hidden)
        hidden = self.drop1(hidden)

        hidden = self.fc2(hidden)
        hidden = self.relu2(hidden)
        hidden = self.drop2(hidden)

        hidden = self.fc3(hidden)
        hidden = self.relu3(hidden)
        hidden = self.drop3(hidden)

        hidden = self.fc4(hidden)
        hidden = self.relu4(hidden)
        hidden = self.drop4(hidden)

        return self.out(hidden)


In [ ]:
%%time
import os, gc, copy
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
from torch.optim import AdamW
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, classification_report
import torch
from torch import nn
import torch.nn.functional as F
from sklearn.metrics import roc_curve, auc
from transformers import AutoTokenizer, get_linear_schedule_with_warmup

# ---------------------------------------------------------------------------
# Funções auxiliares restauradas
# ---------------------------------------------------------------------------
def train_epoch(model, data_loader, loss_fn, optimizer, device, scheduler, n_examples, scaler, use_amp):
    model.train()
    losses, correct_predictions = [], 0
    all_preds, all_targets = [], []
    optimizer.zero_grad(set_to_none=True)

    for d in data_loader:
        input_ids = d["input_ids"].to(device, non_blocking=True)
        attention_mask = d["attention_mask"].to(device, non_blocking=True)
        targets = d["targets"].to(device, non_blocking=True)
        chunk_counts = d["chunk_counts"].to(device, non_blocking=True)

        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, chunk_counts=chunk_counts)
            loss = loss_fn(outputs, targets)

        _, preds = torch.max(outputs, dim=1)
        correct_predictions += torch.sum(preds == targets)
        losses.append(loss.item())
        all_preds.extend(preds.detach().cpu().numpy())
        all_targets.extend(targets.detach().cpu().numpy())

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)

    acc = (correct_predictions.double() / n_examples).item()
    return (acc, np.mean(losses),
            precision_score(all_targets, all_preds, average='macro', zero_division=0),
            recall_score(all_targets, all_preds, average='macro', zero_division=0),
            f1_score(all_targets, all_preds, average='macro', zero_division=0))


def eval_model(model, data_loader, loss_fn, device, n_examples, use_amp=False):
    model.eval()
    losses, correct_predictions = [], 0
    all_preds, all_targets = [], []

    with torch.no_grad():
        for d in data_loader:
            input_ids = d["input_ids"].to(device, non_blocking=True)
            attention_mask = d["attention_mask"].to(device, non_blocking=True)
            targets = d["targets"].to(device, non_blocking=True)
            chunk_counts = d["chunk_counts"].to(device, non_blocking=True)

            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                outputs = model(input_ids=input_ids, attention_mask=attention_mask, chunk_counts=chunk_counts)
                loss = loss_fn(outputs, targets)

            _, preds = torch.max(outputs, dim=1)
            correct_predictions += torch.sum(preds == targets)
            losses.append(loss.item())
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(targets.cpu().numpy())

    acc = (correct_predictions.double() / n_examples).item()
    return (acc, np.mean(losses),
            precision_score(all_targets, all_preds, average='macro', zero_division=0),
            recall_score(all_targets, all_preds, average='macro', zero_division=0),
            f1_score(all_targets, all_preds, average='macro', zero_division=0))


def get_predictions(model, data_loader, device):
    model.eval()
    predictions, prediction_probs, real_values = [], [], []
    with torch.no_grad():
        for d in data_loader:
            input_ids = d["input_ids"].to(device, non_blocking=True)
            attention_mask = d["attention_mask"].to(device, non_blocking=True)
            targets = d["targets"].to(device, non_blocking=True)
            chunk_counts = d["chunk_counts"].to(device, non_blocking=True)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, chunk_counts=chunk_counts)
            _, preds = torch.max(outputs, dim=1)
            predictions.extend(preds)
            prediction_probs.extend(outputs)
            real_values.extend(targets)
    return (torch.stack(predictions).cpu(),
            torch.stack(prediction_probs).cpu(),
            torch.stack(real_values).cpu())


def display_approach_results(y_test, y_pred, probs, fpr, tpr, roc_auc, approach_label, test_acc, test_prec, test_rec, test_f1, class_names):
    print(f"\n{'='*60}")
    print(f"RESULTADOS FINAIS: {approach_label}")
    print(f"{'='*60}")
    print(f"Accuracy:  {test_acc:.4f}")
    print(f"Precision: {test_prec:.4f}")
    print(f"Recall:    {test_rec:.4f}")
    print(f"F1-Score:  {test_f1:.4f}")
    print(f"AUC:       {roc_auc:.4f}\n")

    print("Classification Report:")
    print(classification_report(y_test, y_pred, target_names=class_names, zero_division=0))

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Matriz de Confusão
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
                xticklabels=class_names, yticklabels=class_names)
    axes[0].set_title(f'Matriz de Confusão')
    axes[0].set_ylabel('Real')
    axes[0].set_xlabel('Predito')

    # Curva ROC
    axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'AUC = {roc_auc:.4f}')
    axes[1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    axes[1].set_xlim([0.0, 1.0])
    axes[1].set_ylim([0.0, 1.05])
    axes[1].set_xlabel('False Positive Rate')
    axes[1].set_ylabel('True Positive Rate')
    axes[1].set_title(f'Curva ROC')
    axes[1].legend(loc="lower right")

    plt.tight_layout()
    plt.show()

# ---------------------------------------------------------------------------

model_dir = f'{BASE_PATH_GENERAL}models'
os.makedirs(model_dir, exist_ok=True)

all_roc_data = []
all_metrics = []

tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT, use_fast=True)

for approach_idx, approach_cfg in enumerate(APPROACHES):
    if not approach_cfg.get('enabled', True):
        print(f"\nSkipping approach: {approach_cfg['label']} (disabled)")
        continue

    APPROACH_LABEL = approach_cfg['label']
    APPROACH_STRIDE = approach_cfg['stride']
    APPROACH_MAX_CHUNKS = approach_cfg['max_chunks']
    APPROACH_BATCH_SIZE = approach_cfg['batch_size']
    APPROACH_POOLING = approach_cfg['pooling']
    APPROACH_MIN_CHUNK_TOKENS = approach_cfg.get('min_chunk_tokens', MIN_CHUNK_TOKENS)
    model_save_path = f"{model_dir}/{approach_cfg['model_name']}"
    model_exists = os.path.exists(model_save_path)

    # Lógica de decisão para treinar ou carregar modelo existente
    should_train = FORCE_RETRAIN or not model_exists

    if should_train:
        modo_str = 'treino + inferencia (sobrescrevendo existente)' if model_exists else 'treino + inferencia'
    else:
        modo_str = 'apenas inferencia (carregando modelo existente)'

    print("\n" + "=" * 80)
    print(f"[{approach_idx+1}/{len(APPROACHES)}] {APPROACH_LABEL}")
    print(f"  Modo: {modo_str}")
    print("=" * 80)

    np.random.seed(RANDOM_SEED)
    torch.manual_seed(RANDOM_SEED)

    test_data_loader = create_data_loader(df_test, tokenizer, CHUNK_SIZE, APPROACH_STRIDE, APPROACH_BATCH_SIZE, APPROACH_MAX_CHUNKS, APPROACH_MIN_CHUNK_TOKENS, shuffle=False)
    model = HCAcordaoClassifier(len(class_names), NUM_LAYERS, MODEL_CHECKPOINT, pooling_strategy=APPROACH_POOLING).to(device)
    loss_fn = nn.CrossEntropyLoss().to(device)
    use_amp = (device.type == 'cuda')
    history = defaultdict(list)

    if should_train:
        print(f"  -> Iniciando Fine-Tuning para: {APPROACH_LABEL}")
        train_data_loader = create_data_loader(df_train, tokenizer, CHUNK_SIZE, APPROACH_STRIDE, APPROACH_BATCH_SIZE, APPROACH_MAX_CHUNKS, APPROACH_MIN_CHUNK_TOKENS, shuffle=True)
        optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
        total_steps = len(train_data_loader) * EPOCHS
        scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(total_steps * WARMUP_RATIO), num_training_steps=total_steps)
        scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

        best_recall, best_model_weights = 0, None
        epochs_no_improve = 0

        for epoch in range(EPOCHS):
            train_metrics = train_epoch(model, train_data_loader, loss_fn, optimizer, device, scheduler, len(df_train), scaler, use_amp)
            test_metrics = eval_model(model, test_data_loader, loss_fn, device, len(df_test), use_amp)

            print(f"Epoch {epoch+1:03d}/{EPOCHS} | "
                  f"Train -> Acc: {train_metrics[0]:.4f}, Prec: {train_metrics[2]:.4f}, Rec: {train_metrics[3]:.4f}, F1: {train_metrics[4]:.4f} | "
                  f"Val -> Acc: {test_metrics[0]:.4f}, Prec: {test_metrics[2]:.4f}, Rec: {test_metrics[3]:.4f}, F1: {test_metrics[4]:.4f}")

            if test_metrics[3] > best_recall: # index 3 is recall
                best_model_weights = copy.deepcopy(model.state_dict())
                best_recall = test_metrics[3]
                epochs_no_improve = 0
            else:
                epochs_no_improve += 1
                if epochs_no_improve >= EARLY_STOPPING:
                    print(f"  -> Early stopping triggered at epoch {epoch+1}!")
                    break

        if best_model_weights:
            model.load_state_dict(best_model_weights)
            torch.save(best_model_weights, model_save_path)

    else:
        print(f"  -> Carregando modelo existente: {model_save_path}")
        model.load_state_dict(torch.load(model_save_path, map_location=device))

    # Coleta métricas finais...
    test_acc, test_loss, test_prec, test_rec, test_f1 = eval_model(model, test_data_loader, loss_fn, device, len(df_test), use_amp)
    y_pred, y_pred_probs, y_test = get_predictions(model, test_data_loader, device)
    probs = F.softmax(y_pred_probs, dim=1).numpy()
    fpr, tpr, _ = roc_curve(y_test.numpy(), probs[:, 1])
    roc_auc = auc(fpr, tpr)

    # Exibe os resultados finais com gráficos para a abordagem atual
    display_approach_results(y_test.numpy(), y_pred.numpy(), probs, fpr, tpr, roc_auc, APPROACH_LABEL, test_acc, test_prec, test_rec, test_f1, class_names)

    all_roc_data.append((approach_idx + 1, APPROACH_LABEL, fpr, tpr, roc_auc))
    all_metrics.append({'ID': approach_idx + 1, 'Approach': APPROACH_LABEL, 'Accuracy': test_acc, 'Precision': test_prec, 'Recall': test_rec, 'F1-Score': test_f1, 'AUC': roc_auc})

    del model; gc.collect(); torch.cuda.empty_cache() if device.type == 'cuda' else None


## Comparação Final — Curva ROC

In [ ]:
plt.figure(figsize=(10, 8))
colors = ['#FF006D', '#01BEFE', '#FFDD00', '#8F00FF', '#ADFF02', '#FF7D00']

for i, (approach_id, label, fpr, tpr, roc_auc) in enumerate(all_roc_data):
    plt.plot(fpr, tpr, color=colors[i % len(colors)], lw=2.5,
             label=f'[{approach_id}] {label} (AUC = {roc_auc:.4f})')

plt.plot([0, 1], [0, 1], 'navy', lw=2, linestyle='--', label='Random')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('Curva ROC Comparativa — 7 Abordagens', fontsize=14)
plt.legend(loc="lower right", fontsize=11)
plt.grid(alpha=0.3, linestyle='--')
plt.tight_layout()
plt.show()

## Tabela Comparativa de Metricas

In [ ]:
df_metrics = pd.DataFrame(all_metrics)
df_metrics = df_metrics.sort_values('ID', ascending=True).reset_index(drop=True)

print("=" * 80)
print("TABELA COMPARATIVA — ABORDAGENS (Truncation | Mean/Max Pooling × Stride 0/128/256)")
print("=" * 80)
display(
    df_metrics.style
    .highlight_max(subset=['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC'], color='lightgreen')
)
